# Derm-Vision: ISIC 2019 Skin Lesion Classification — Training

This notebook runs the full training pipeline on Google Colab with GPU.

Data is downloaded directly from Kaggle to Colab's local disk — no Google Drive storage needed for the dataset.

## 1. GPU Check

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=memory.total --format=csv,noheader

## 2. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Lambert-Nguyen/derm-vision.git /content/derm-vision
%cd /content/derm-vision

In [ ]:
!pip install -q timm albumentations wandb grad-cam kaggle ttach

## 3. Download ISIC 2019 from Kaggle

Downloads directly to Colab's local disk (~100GB free) — no Drive space needed.

**Setup**: Go to https://www.kaggle.com/settings → copy your **Username** and **API Key**, then paste them below.

In [ ]:
import os
import json
from getpass import getpass

# Prompt for credentials (won't be displayed or saved in the notebook)
kaggle_username = input("Kaggle username: ")
kaggle_key = getpass("Kaggle API key: ")

# Write kaggle.json
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(kaggle_json_path, 0o600)
print("Kaggle API configured.")

In [ ]:
# Download ISIC 2019 dataset
!kaggle datasets download -d andrewmvd/isic-2019 -p /content/isic-data --unzip

# List downloaded files
!ls /content/isic-data/

In [ ]:
# Symlink data into the project directory
import os
import glob

os.makedirs("data/raw", exist_ok=True)

# Auto-detect the data layout from Kaggle
kaggle_dir = "/content/isic-data"

# Find the images directory
img_dir = None
for candidate in [
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input"),
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input"),
]:
    if os.path.isdir(candidate) and glob.glob(os.path.join(candidate, "*.jpg")):
        img_dir = candidate
        break

if img_dir is None:
    raise FileNotFoundError(
        f"Could not find images in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}"
    )

# Find CSV files
gt_matches = glob.glob(os.path.join(kaggle_dir, "**", "*GroundTruth*"), recursive=True)
meta_matches = glob.glob(os.path.join(kaggle_dir, "**", "*Metadata*"), recursive=True)
if not gt_matches or not meta_matches:
    raise FileNotFoundError(f"Missing CSVs in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}")
gt_csv = gt_matches[0]
meta_csv = meta_matches[0]

# Create symlinks
links = [
    (img_dir, "data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input"),
    (gt_csv, "data/raw/ISIC_2019_Training_GroundTruth.csv"),
    (meta_csv, "data/raw/ISIC_2019_Training_Metadata.csv"),
]

for src, dst in links:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"{dst} -> {src}")

# Verify
n_images = len(glob.glob("data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input/*.jpg"))
print(f"\nFound {n_images} images.")

## 4. Create Data Splits

In [ ]:
if not os.path.exists("data/splits/train.csv"):
    !python scripts/create_splits.py
else:
    print("Splits already exist, skipping.")

## 5. Mount Drive for Checkpoints Only

Checkpoints (~50MB) are saved to Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import yaml

# Read config, update output paths only, rewrite preserving structure
config_path = "configs/config.yaml"
with open(config_path) as f:
    config_text = f.read()
    cfg = yaml.safe_load(config_text)

# Save checkpoints & results to Drive (small files only)
drive_output = "/content/drive/MyDrive/derm-vision-outputs"
os.makedirs(os.path.join(drive_output, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(drive_output, "results"), exist_ok=True)

# Replace only the output paths via string replacement to preserve comments
config_text = config_text.replace(
    f"checkpoint_dir: {cfg['output']['checkpoint_dir']}",
    f"checkpoint_dir: {os.path.join(drive_output, 'checkpoints')}"
)
config_text = config_text.replace(
    f"results_dir: {cfg['output']['results_dir']}",
    f"results_dir: {os.path.join(drive_output, 'results')}"
)

with open(config_path, "w") as f:
    f.write(config_text)

# Reload to verify
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print("Checkpoints will save to Drive (minimal storage needed).")
print(f"  checkpoint_dir: {cfg['output']['checkpoint_dir']}")
print(f"  results_dir: {cfg['output']['results_dir']}")

## 6. Login to W&B

In [ ]:
import wandb
wandb.login()

## 7. Train

In [ ]:
!python -m src.train --config configs/config.yaml

## 8. Evaluate on Test Set (with Test-Time Augmentation)

In [ ]:
import ttach
import yaml
import torch
from src.dataset import ISICDataset, CLASS_NAMES
from src.transforms import get_val_transforms
from src.models.efficientnet import EfficientNetB3Classifier
from src.evaluate import compute_metrics, print_classification_report, plot_confusion_matrix
from torch.utils.data import DataLoader

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model
model = EfficientNetB3Classifier(
    num_classes=cfg["data"]["num_classes"],
    pretrained=False,
    dropout=cfg["model"]["dropout"],
).to(device)
ckpt_path = os.path.join(cfg["output"]["checkpoint_dir"], "best_model.pth")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

# Wrap with test-time augmentation (flips + rotations, averaged)
tta_model = ttach.ClassificationTTAWrapper(
    model,
    ttach.aliases.d4_transform(),  # 8 variants: flips + 90° rotations
    merge_mode="mean",
)

# Test dataset
test_dataset = ISICDataset(
    image_dir=cfg["data"]["data_dir"],
    labels_csv=os.path.join(cfg["data"]["splits_dir"], "test.csv"),
    transform=get_val_transforms(cfg["data"]["image_size"]),
)
test_loader = DataLoader(test_dataset, batch_size=cfg["training"]["batch_size"], num_workers=2)

# --- Evaluate WITH TTA ---
print("=" * 50)
print("Results WITH Test-Time Augmentation (D4)")
print("=" * 50)
all_preds_tta, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = tta_model(images.to(device))
        all_preds_tta.extend(outputs.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

metrics_tta = compute_metrics(all_labels, all_preds_tta)
print(f"Balanced Accuracy: {metrics_tta['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics_tta['weighted_f1']:.4f}")
print()
print_classification_report(all_labels, all_preds_tta)

save_path = os.path.join(cfg["output"]["results_dir"], "confusion_matrix_tta.png")
plot_confusion_matrix(all_labels, all_preds_tta, save_path=save_path)

# --- Evaluate WITHOUT TTA (for comparison) ---
print("\n" + "=" * 50)
print("Results WITHOUT TTA (baseline)")
print("=" * 50)
all_preds_base = []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds_base.extend(outputs.argmax(dim=1).cpu().tolist())

metrics_base = compute_metrics(all_labels, all_preds_base)
print(f"Balanced Accuracy: {metrics_base['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics_base['weighted_f1']:.4f}")

# Summary
print("\n" + "=" * 50)
print("TTA Improvement")
print("=" * 50)
print(f"Balanced Accuracy: {metrics_base['balanced_accuracy']:.4f} → {metrics_tta['balanced_accuracy']:.4f} ({metrics_tta['balanced_accuracy'] - metrics_base['balanced_accuracy']:+.4f})")
print(f"Weighted F1:       {metrics_base['weighted_f1']:.4f} → {metrics_tta['weighted_f1']:.4f} ({metrics_tta['weighted_f1'] - metrics_base['weighted_f1']:+.4f})")

## 9. Train Custom CNN Baseline

Train a simple 4-layer CNN from scratch for comparison against EfficientNet-B3.

In [ ]:
import yaml

# Update CNN config to use Drive output paths
cnn_config_path = "configs/config_cnn.yaml"
with open(cnn_config_path) as f:
    cnn_text = f.read()
    cnn_cfg = yaml.safe_load(cnn_text)

drive_output = "/content/drive/MyDrive/derm-vision-outputs"
cnn_text = cnn_text.replace(
    f"checkpoint_dir: {cnn_cfg['output']['checkpoint_dir']}",
    f"checkpoint_dir: {os.path.join(drive_output, 'checkpoints_cnn')}"
)
cnn_text = cnn_text.replace(
    f"results_dir: {cnn_cfg['output']['results_dir']}",
    f"results_dir: {os.path.join(drive_output, 'results_cnn')}"
)

with open(cnn_config_path, "w") as f:
    f.write(cnn_text)

os.makedirs(os.path.join(drive_output, "checkpoints_cnn"), exist_ok=True)
os.makedirs(os.path.join(drive_output, "results_cnn"), exist_ok=True)

!python -m src.train --config configs/config_cnn.yaml

## 10. Evaluate Custom CNN on Test Set

In [ ]:
import yaml
import torch
from src.dataset import ISICDataset, CLASS_NAMES
from src.transforms import get_val_transforms
from src.models.custom_cnn import CustomCNN
from src.evaluate import compute_metrics, print_classification_report, plot_confusion_matrix
from torch.utils.data import DataLoader

with open("configs/config_cnn.yaml") as f:
    cnn_cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best CNN model
cnn_model = CustomCNN(
    num_classes=cnn_cfg["data"]["num_classes"],
    dropout=cnn_cfg["model"]["dropout"],
).to(device)
cnn_ckpt = os.path.join(cnn_cfg["output"]["checkpoint_dir"], "best_model.pth")
cnn_model.load_state_dict(torch.load(cnn_ckpt, map_location=device))
cnn_model.eval()

# Test dataset (224px for CNN)
cnn_test_dataset = ISICDataset(
    image_dir=cnn_cfg["data"]["data_dir"],
    labels_csv=os.path.join(cnn_cfg["data"]["splits_dir"], "test.csv"),
    transform=get_val_transforms(cnn_cfg["data"]["image_size"]),
)
cnn_test_loader = DataLoader(cnn_test_dataset, batch_size=cnn_cfg["training"]["batch_size"], num_workers=2)

# Run inference
cnn_preds, cnn_labels = [], []
with torch.no_grad():
    for images, labels in cnn_test_loader:
        outputs = cnn_model(images.to(device))
        cnn_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        cnn_labels.extend(labels.tolist())

cnn_metrics = compute_metrics(cnn_labels, cnn_preds)
print("Custom CNN Results")
print("=" * 50)
print(f"Balanced Accuracy: {cnn_metrics['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {cnn_metrics['weighted_f1']:.4f}")
print()
print_classification_report(cnn_labels, cnn_preds)

save_path = os.path.join(cnn_cfg["output"]["results_dir"], "confusion_matrix_cnn.png")
plot_confusion_matrix(cnn_labels, cnn_preds, save_path=save_path)

## 11. Model Comparison Summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.dataset import CLASS_NAMES

# Collect per-class F1 from classification reports
from sklearn.metrics import f1_score, classification_report

# EfficientNet + TTA metrics (from cell 8)
eff_report = classification_report(all_labels, all_preds_tta, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
# CNN metrics (from cell 10)
cnn_report = classification_report(cnn_labels, cnn_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

# --- Summary Table ---
summary = {
    "Metric": ["Balanced Accuracy", "Weighted F1", "Accuracy"],
    "Custom CNN": [
        f"{cnn_metrics['balanced_accuracy']:.4f}",
        f"{cnn_metrics['weighted_f1']:.4f}",
        f"{cnn_report['accuracy']:.4f}",
    ],
    "EfficientNet-B3 + TTA": [
        f"{metrics_tta['balanced_accuracy']:.4f}",
        f"{metrics_tta['weighted_f1']:.4f}",
        f"{eff_report['accuracy']:.4f}",
    ],
}
summary_df = pd.DataFrame(summary)
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(summary_df.to_string(index=False))

# --- Per-class F1 comparison ---
print("\n" + "=" * 60)
print("PER-CLASS F1 COMPARISON")
print("=" * 60)
per_class = {"Class": CLASS_NAMES}
per_class["CNN F1"] = [f"{cnn_report[c]['f1-score']:.2f}" for c in CLASS_NAMES]
per_class["EfficientNet+TTA F1"] = [f"{eff_report[c]['f1-score']:.2f}" for c in CLASS_NAMES]
per_class["Support"] = [cnn_report[c]["support"] for c in CLASS_NAMES]
per_class_df = pd.DataFrame(per_class)
print(per_class_df.to_string(index=False))

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(CLASS_NAMES))
width = 0.35
cnn_f1s = [cnn_report[c]["f1-score"] for c in CLASS_NAMES]
eff_f1s = [eff_report[c]["f1-score"] for c in CLASS_NAMES]

bars1 = ax.bar([i - width/2 for i in x], cnn_f1s, width, label="Custom CNN", color="#e74c3c", alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], eff_f1s, width, label="EfficientNet-B3 + TTA", color="#2ecc71", alpha=0.8)

ax.set_xlabel("Class")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Score: Custom CNN vs EfficientNet-B3 + TTA")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_path = os.path.join(cfg["output"]["results_dir"], "model_comparison.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved to {save_path}")